# 3.5 分數為什麼需要縮放？

# 第 3 章：Attention，自己決定看哪裡

前置：向量、矩陣乘法和 softmax。先用單 head；每個小節都畫得出矩陣。完整模型仍留到第4章。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：把分數換成比例：像分一塊餅**

一起加同常數不改比例，分數差才重要。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/softmax.svg")))

**先想一想：**D 變為4倍時，標準差大約怎麼變？

相加的 feature 越多，隨機點積通常波動越大，softmax 容易過早變尖。除以平方根維度讓尺度比較穩定。

**把直覺寫成數學：**獨立零均值項相加的 variance 約為 D，standard deviation 約為 √D；因此用 $QK^T/\sqrt D$。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
import math

for d in (4, 64):
    q, k = torch.randn(1000, d), torch.randn(1000, d)
    scores = (q * k).sum(-1)
    print(d, scores.std().item(), (scores / math.sqrt(d)).std().item())

**如何讀結果：**是隨機獨立假設下的尺度直覺，不是每個已訓練向量的嚴格保證。

**只改一件事：**只把兩組向量的標準差加倍。
